### Annotate protein matches with pairwise BLAST
### Julian Moran
### 2026-09-25

In [17]:
import boto3
import io
import glob
import logging
import math
import os
import requests
import s3fs
import subprocess
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv
from pathlib import Path

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# ============================================================
#       Args
# ============================================================

# API endpoints


# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.manifest.json',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [32]:
# ============================================================
#       In
# ============================================================

df_seqs_hs = pl.read_csv(
    f"{REPO_ROOT}/results/iei_human_protein_AAs.tsv",
    separator="\t",
    has_header=True
)

df_seqs_bact = pl.read_csv(
    f"{REPO_ROOT}/results/iei_bact_protein_AAs.tsv",
    separator="\t",
    has_header=True
)

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)

df_comp_score = df_comp_score.rename(
    {
        "defense_uniprot_ac": "bact_uniprot",
        "human_entryId": "hs_uniprot"
    }
).sort("composite_score", descending=True)
df_comp_score

bact_uniprot,hs_uniprot,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""Q47ZY9""","""Q8IZH9""",0.9962,0.0,0.28,0.991,0.964,95.529999,97.120003
"""A0A5B9E1W6""","""Q5QP19""",0.9863,0.0,0.54,0.9237,0.9496,94.519997,95.110001
"""A0A1D7QWY7""","""Q5QP19""",0.986,0.0,0.58,0.9202,0.9533,94.519997,95.519997
"""A0A5B9QMN2""","""Q5QP19""",0.9852,0.0,0.58,0.9126,0.9526,94.519997,94.910004
"""Q16C13""","""Q5QP19""",0.9838,0.0,0.7,0.9233,0.9467,94.519997,95.300003
…,…,…,…,…,…,…,…,…
"""C3N4E9""","""A0A024QZ47""",0.1357,0.03661,null,null,null,null,61.529999
"""Q97YA9""","""A0A024QZ47""",0.1351,0.03661,null,null,null,null,60.720001
"""A0A0F7P5S5""","""A0A024R2W4""",0.1338,0.03729,null,null,null,null,59.610001


In [33]:
df_seqs_hs

uniprot_accession,sequence
str,str
"""Q16637""","""MAMSSGGSGGGVPEQEDSVLFRRGTGQSDD…"
"""Q8IY37""","""MGKLRRRYNIKGRQQAGPGPSKGPPEPPPV…"
"""Q8NB16""","""MENLKHIITLGQVIHKRCEEMKYCKKQCRR…"
"""Q9NZ01""","""MKHYEVEILDAKTREKLCFLDKVEPHATIA…"
"""Q02156""","""MVVFNGLLKIKICEAVSLKPTAWSLRHAVG…"
…,…
"""J7F7B0""","""SHSMRYFYTAMSRPGRGEPRFITVGYVDDT…"
"""I6MHI2""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""D3DP96""","""MKERRASQKLSSKSIMDPNQNVKCKIVVVG…"


In [34]:
# ============================================================
#       Wrangle
# ============================================================

df_seqs = (
    df_comp_score
    .join(
        df_seqs_bact.rename({
            "uniprot_accession": "bact_uniprot",
            "sequence": "bact_seq"
        }),
        on="bact_uniprot",
        how="left"
    )
    .join(
        df_seqs_hs.rename({
            "uniprot_accession": "hs_uniprot",
            "sequence": "hs_seq"
        }),
        on="hs_uniprot",
        how="left"
    )
    .select([
        "bact_uniprot",
        "hs_uniprot",
        "bact_seq",
        "hs_seq"
    ])
)
df_seqs

bact_uniprot,hs_uniprot,bact_seq,hs_seq
str,str,str,str
"""Q47ZY9""","""Q8IZH9""","""MTIENNYREILATIGEDINRGGLLDTPKRA…","""AYSSILSSLGENPQRQGLLKTPWRAASAMQ…"
"""A0A5B9E1W6""","""Q5QP19""","""MSDGRPIYLDHNATTPLDPTVFEAMRPYFL…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…"
"""A0A1D7QWY7""","""Q5QP19""","""MIYFDNSSTTPLHPEVKEAMWPYINEEFGN…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…"
"""A0A5B9QMN2""","""Q5QP19""","""MPPVYLDYNATTPLDPRVFEVMKEWYLGPP…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…"
"""Q16C13""","""Q5QP19""","""MSAPIYLDHNASTPIDPEVLETVIRVSRDV…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…"
…,…,…,…
"""C3N4E9""","""A0A024QZ47""","""MVTIIASDLVYLSYELENYTKNVNKAIICG…","""MAAVELEWIPETLYNTAISAVVDNYIRSRR…"
"""Q97YA9""","""A0A024QZ47""","""MVYLNYEIENYTRNVKRVIVCGDVKGFRLR…","""MAAVELEWIPETLYNTAISAVVDNYIRSRR…"
"""A0A0F7P5S5""","""A0A024R2W4""","""MVSDGRDGDGRIDDSEFEQNKEPISQDIAN…","""MRMSVGLSLLLPLWGRTFLLLLSVVMAQSH…"


In [37]:
# ============================================================
#       BLAST
# ============================================================

def pairwise_blastp(
    query: str,
    query_ac: str,
    subject: str,
    subject_ac: str
) -> pl.DataFrame | list[pl.DataFrame]:

    with tempfile.TemporaryDirectory() as tmpdir:
        query_path = Path(tmpdir) / "query.fasta"
        subject_path = Path(tmpdir) / "subject.fasta"

        query_path.write_text(f">{query_ac}\n{query}\n")
        subject_path.write_text(f">{subject_ac}\n{subject}\n")
        result = subprocess.run(
            [
                "blastp",
                "-query", str(query_path),
                "-subject", str(subject_path),
                "-outfmt", "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore",
            ],
            capture_output=True,
            text=True,
            check=True,
        )
        columns = [
            "query_ac",
            "subject_ac",
            "pident",
            "length",
            "mismatch",
            "gapopen",
            "qstart",
            "qend",
            "sstart",
            "send",
            "evalue",
            "bitscore",
        ]
        if not result.stdout.strip():
            return pl.DataFrame([
                {
                    column: query_ac if column == "query_ac"
                    else subject_ac if column == "subject_ac"
                    else None
                    for column in columns
                }
            ])
        df_blast = pl.read_csv(
            io.StringIO(result.stdout),
            separator="\t",
            has_header=False,
            new_columns=columns,
            infer_schema=False
        ).sort("bitscore", descending=True)
        return df_blast[0]

def pairwise_blastp_iter(
    df_seqs: pl.DataFrame
) -> pl.DataFrame:
    results = []
    for i in range(len(df_seqs)):
        try:
            result = pairwise_blastp(
                query=df_seqs["bact_seq"][i],
                query_ac=df_seqs["bact_uniprot"][i],
                subject=df_seqs["hs_seq"][i],
                subject_ac=df_seqs["hs_uniprot"][i]
            )
            results.append(result)
        except Exception as e:
            logger.warning(
                f"BLASTP failed for row {i} "
                f"({df_seqs['bact_uniprot'][i]} vs "
                f"{df_seqs['hs_uniprot'][i]}): {e}"
            )
    try:
        df_blast = pl.concat(results, how="vertical_relaxed")
    except Exception as e:
        logger.warning(f"BLASTp results concatenation failed: {e}")
        return results
    df_ann = df_seqs.join(
        df_blast,
        left_on=["bact_uniprot", "hs_uniprot"],
        right_on=["query_ac", "subject_ac"],
        how="left"
    )
    return df_ann

df_blast = pairwise_blastp_iter(df_seqs[:10000])
df_blast

bact_uniprot,hs_uniprot,bact_seq,hs_seq,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Q47ZY9""","""Q8IZH9""","""MTIENNYREILATIGEDINRGGLLDTPKRA…","""AYSSILSSLGENPQRQGLLKTPWRAASAMQ…","""56.000""","""175""","""77""","""0""","""7""","""181""","""2""","""176""","""7.30e-82""","""227"""
"""A0A5B9E1W6""","""Q5QP19""","""MSDGRPIYLDHNATTPLDPTVFEAMRPYFL…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…","""44.180""","""378""","""208""","""3""","""9""","""385""","""1""","""376""","""5.37e-114""","""326"""
"""A0A1D7QWY7""","""Q5QP19""","""MIYFDNSSTTPLHPEVKEAMWPYINEEFGN…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…","""39.410""","""373""","""215""","""6""","""4""","""373""","""1""","""365""","""7.46e-98""","""285"""
"""A0A5B9QMN2""","""Q5QP19""","""MPPVYLDYNATTPLDPRVFEVMKEWYLGPP…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…","""45.225""","""356""","""187""","""6""","""6""","""358""","""1""","""351""","""2.09e-102""","""296"""
"""Q16C13""","""Q5QP19""","""MSAPIYLDHNASTPIDPEVLETVIRVSRDV…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…","""38.005""","""371""","""226""","""4""","""7""","""376""","""1""","""368""","""1.20e-87""","""259"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q4FUP0""","""B7Z1Y7""","""MKYIELFAGCGGLSLGLQAVGFENIMVNEL…","""MILMSPPCQPFTRIGRQGDMTDSRTNSFLY…","""34.000""","""50""","""26""","""2""","""136""","""180""","""5""","""52""","""0.002""","""25.4"""
"""K9VWY1""","""A0A140VJS3""","""MNKTERLKNLTLPCTPTLLVTALTLFLGGG…","""MALDGPEQMELEEGKAGSGLRQYYLSKIEE…","""24.779""","""113""","""67""","""5""","""342""","""440""","""104""","""212""","""0.38""","""19.2"""
"""A0A4Y6UVB6""","""J3QLN6""","""MSAEHPFYRLAPFIQEYIYRSGWEELREVQ…","""MNLSESLLRGIYAYGFEKPSAIQQRAILPC…","""37.500""","""16""","""10""","""0""","""244""","""259""","""147""","""162""","""3.0""","""15.0"""


In [38]:
# ============================================================
#       Out
# ============================================================

df_blast.write_csv(
    f"{REPO_ROOT}/results/iei-pipeline_blasts_n=10000.tsv",
    separator="\t"
)

In [ ]:
# ============================================================
#       Wrangle
# ============================================================


